# 11.2 Preparing Text Data

Vocabulary Indexing

Standardization is the processing of converting to lowercase, removing punctuations and non ascii characters e cap. But remember removing punctuations is not always optimal. For example, if you want to find questions asked in an interview from its transcript, you must include ```'?'``` character.

    vocabulary = {}
    for text in dataset:
        text = standardize(text)
        tokens = tokenize(text)
        for token in tokens:
            if token not in vocabulary:
                vocabulary[tokens] = len(vocabulary)

After this, we can do one hot encoding:

    def one_hot_encode_token(token):
        vector = np.zeros((len(vocabulary), ))
        vector[vocabulary[token]] = 1
        return vector

In [1]:
import string

class Vectorizer:
    def standardize(self, text):
        text = text.lower()
        return "".join(char for char in text
                       if char not in string.punctuation)

    def tokenize(self, text):
        text = self.standardize(text);
        return text.split()

    def make_vocabulary(self, dataset):
        self.vocabulary = {"": 0,
                           "[UNK]": 1}
        for text in dataset:
            text = self.standardize(text)
            tokens = self.tokenize(text)
            for token in tokens:
                if token not in self.vocabulary:
                    self.vocabulary[token] = len(self.vocabulary)

        self.inverse_vocabulary = dict((v, k) for k, v in self.vocabulary.items())

    def encode(self, text):
        text = self.standardize(text)
        tokens = self.tokenize(text)
        return [self.vocabulary.get(token, 1) for token in tokens]

    def decode(self, int_sequence):
        return " ".join(self.inverse_vocabulary.get(i, "[UNK]") for i in int_sequence)

vectorizer = Vectorizer()
dataset = [
    "I write, erase, rewrite",
    "Erase again, and then",
    "A poppy blooms.",
]

vectorizer.make_vocabulary(dataset)

test_sentence = "I write, rewrite, and still rewrite again."
encoded_sentence = vectorizer.encode(test_sentence)
print(encoded_sentence)
decoded_sentence = vectorizer.decode(encoded_sentence)
print(decoded_sentence)

[2, 3, 5, 7, 1, 5, 6]
i write rewrite and [UNK] rewrite again


In [2]:
from tensorflow.keras.layers import TextVectorization
text_vectorization = TextVectorization(
    output_mode="int"
)

By default the text vectorization layer from keras does standardization by lowercasing the text and removing punctuations. This behaviour can be changed by implementing our own custom standardization functions, provided that the work on tf.string and not on python strings!

In [3]:
import re
import string
import tensorflow as tf

def custom_standardization_fn(string_tensor):
    lowercase_string = tf.strings.lower(string_tensor)
    return tf.strings.regex_replace(
        lowercase_string, f"[{re.escape(string.punctuation)}]", ""
    )

def custom_split_function(string_tensor):
    return tf.strings.split(string_tensor)

text_vectorization = TextVectorization(
    output_mode='int',
    standardize=custom_standardization_fn,
    split=custom_split_function,
)

text_vectorization.adapt(dataset)

print(tf.string) # This is a datatype
print(tf.strings) # This is the library to operate on the above mentioned data type

<dtype: 'string'>
<module 'tensorflow._api.v2.strings' from '/usr/local/lib/python3.11/dist-packages/tensorflow/_api/v2/strings/__init__.py'>


In [4]:
voc = text_vectorization.get_vocabulary()
print(type(voc))
print([type(s) for s in voc])

<class 'list'>
[<class 'str'>, <class 'str'>, <class 'numpy.str_'>, <class 'numpy.str_'>, <class 'numpy.str_'>, <class 'numpy.str_'>, <class 'numpy.str_'>, <class 'numpy.str_'>, <class 'numpy.str_'>, <class 'numpy.str_'>, <class 'numpy.str_'>, <class 'numpy.str_'>]


In [5]:
text_vectorization(test_sentence).numpy()


array([ 7,  3,  5,  9,  1,  5, 10])

# 11.3 Two approaches for representing groups of words: Sets and sequences

In [6]:
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  19.4M      0  0:00:04  0:00:04 --:--:-- 19.4M


In [7]:
!rm -r aclImdb/train/unsup/

In [8]:
!cat aclImdb/train/pos/1103_10.txt
print()
!cat aclImdb/train/neg/11050_1.txt

This film has some of the greatest comedic dialog and memorable quotes ever assembled in one film! The plot is somewhat lacking, but the delightful quips are enough to make up the difference. This is a timeless movie for all ages that is sure to please. As a cinematic art form it is highly entertaining; and with major stars like Cary Grant, Myrna Loy, and Melvyn Douglas... how could you go wrong? <br /><br />Comedic dialog and timeing such as this has long been undervalued, and is very difficult to imitate. A good example of this is seen in the 1986 knockoff of this film: The Money Pit, with Tom Hanks and Shelley Long. Despite the talent and physical comedy of these stars, the film dragged and received poor reviews and viewer comments. Achieving true comedic dialog is an art.
Terrible use of scene cuts. All continuity is lost, either by awful scripting or lethargic direction. That villainous robot... musta been a jazz dancer? Also, one of the worst sound tracks I've ever heard (monolog

In [9]:
import os, pathlib, shutil, random

base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"
train_dir = base_dir / "train"

for category in ("neg", "pos"):
    os.makedirs(val_dir / category)
    files = os.listdir(train_dir / category)
    # print(type(files))
    random.Random(1337).shuffle(files)
    num_val_samples = int (0.2 * len(files))
    val_files = files[-num_val_samples: ]

    for fname in val_files:
        shutil.move(train_dir / category / fname,
                    val_dir / category / fname)

In [10]:
from tensorflow import keras

batch_size = 32

train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train",
    batch_size=batch_size
)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val",
    batch_size=batch_size
)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test",
    batch_size=batch_size
)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


By running the following two blocks various times, we can see that the dataset object by default is random in order

In [11]:
for data in train_ds:
    print(type(data))
    print(data[0][0])
    print(data[1][0])
    break

<class 'tuple'>
tf.Tensor(b"I like movies about morally corrupt characters, but this was too much. The acting wasn't great, but that wasn't the real problem. The issue was the sinking feeling I got in the pit of my stomach about 20 minutes into the film. These characters were hollow. They had almost no depth, and what little they did have was devoted to the cruelty they displayed to each other in the guise of friendship. Exploring the darker sides of a set of characters can be fascinating, but you have to give those characters actual personalities or they are just cardboard cutouts. These characters were cardboard and the picture they gave was just ugly.", shape=(), dtype=string)
tf.Tensor(0, shape=(), dtype=int32)


In [12]:
for inputs, targets in train_ds:
    print("inputs.shape: ", inputs.shape)
    print("inputs.dtype: ", inputs.dtype)
    print("targets.shape: ", targets.shape)
    print("targets.dtype: ", targets.dtype)
    print("inputs[0]: ", inputs[0])
    print("targets[0]: ", targets[0])
    break

inputs.shape:  (32,)
inputs.dtype:  <dtype: 'string'>
targets.shape:  (32,)
targets.dtype:  <dtype: 'int32'>
inputs[0]:  tf.Tensor(b'In all truth, this really isn\'t a "movie" so much as an extended final episode; by this I mean that, had you NOT followed the TV series (Homicide: Life On The Street) I suspect that you would have a hard time following this made-for-tv movie. Having said that, "Homicide: The Movie" is still a great watch. I think it says a lot about a television production that EVERY single cast member would return, many after years of absence, to once again portray their characters and bring closure to an incredible program. The movie brings out that sense of "family", not only amongst the characters, but amongst the actors, as well. It\'s all very bitter-sweet knowing that this will be the LAST time we will see them all together again under the title of HOMICIDE. Story-wise, I found this film somewhat lacking. Giardello\'s mayoral candidacy seems particularly contrived

## 11.3.2 Processing words as a set: The bag-of-words approach

In [13]:
text_vectorization = TextVectorization(
    max_tokens=20000,
    output_mode="multi_hot",
)

text_only_train_ds = train_ds.map(lambda x, y: x)
text_vectorization.adapt(text_only_train_ds)

binary_1gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
binary_1gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
binary_1gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)


In [14]:
for inputs, targets in binary_1gram_train_ds:
    print("inputs.shape: ", inputs.shape)
    print("inputs.dtype: ", inputs.dtype)
    print("inputs[0]: ", inputs[0])
    print("targets.shape: ", targets.shape)
    print("targets.dtype: ", targets.dtype)
    print("targets[0]: ", targets[0])
    # print()
    break

inputs.shape:  (32, 20000)
inputs.dtype:  <dtype: 'int64'>
inputs[0]:  tf.Tensor([1 1 1 ... 0 0 0], shape=(20000,), dtype=int64)
targets.shape:  (32,)
targets.dtype:  <dtype: 'int32'>
targets[0]:  tf.Tensor(0, shape=(), dtype=int32)


In [15]:
from tensorflow import keras
from tensorflow.keras import layers

def get_model(max_tokens=20000, hidden_dim=16):
    inputs = keras.Input(shape=(max_tokens, ))
    x = layers.Dense(hidden_dim, activation='relu')(inputs)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer='rmsprop',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [16]:
model = get_model()
model.summary()

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='binary_1gram.keras',
        save_best_only=True
    )
]

model.fit(
    binary_1gram_train_ds.cache(),
    validation_data=binary_1gram_val_ds.cache(),
    epochs=10,
    callbacks=callbacks
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - accuracy: 0.7685 - loss: 0.4965 - val_accuracy: 0.8866 - val_loss: 0.2951
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8886 - loss: 0.2874 - val_accuracy: 0.8816 - val_loss: 0.3036
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9146 - loss: 0.2438 - val_accuracy: 0.8858 - val_loss: 0.3077
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9213 - loss: 0.2232 - val_accuracy: 0.8854 - val_loss: 0.3184
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9257 - loss: 0.2236 - val_accuracy: 0.8852 - val_loss: 0.3324
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9290 - loss: 0.2061 - val_accuracy: 0.8836 - val_loss: 0.3493
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9369 - loss: 0.1994 - val_accuracy: 0.8848 - val_loss: 0.3539
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9384 - loss: 0.2063 - val_accuracy: 0.

In [17]:
model = keras.models.load_model('binary_1gram.keras')
print(f"Test Accuracy: {model.evaluate(binary_1gram_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8837 - loss: 0.3020
Test Accuracy: 0.884


In [18]:
text_vectorization = TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode='multi_hot'
)

text_vectorization.adapt(text_only_train_ds)

In [19]:
binary_2gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
binary_2gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
binary_2gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)

model = get_model()
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='binary_2gram.keras',
        save_best_only=True
    )
]

model.fit(
    binary_2gram_train_ds.cache(),
    validation_data=binary_2gram_val_ds.cache(),
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.7811 - loss: 0.4633 - val_accuracy: 0.8920 - val_loss: 0.2800
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9114 - loss: 0.2448 - val_accuracy: 0.8990 - val_loss: 0.2815
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9340 - loss: 0.2022 - val_accuracy: 0.8980 - val_loss: 0.2962
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9418 - loss: 0.1866 - val_accuracy: 0.8986 - val_loss: 0.3247
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9460 - loss: 0.1820 - val_accuracy: 0.8974 - val_loss: 0.3314
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9503 - loss: 0.1686 - val_accuracy: 0.8970 - val_loss: 0.3568
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9554 - loss: 0.1619 - val_accuracy: 0.8934 - val_loss: 0.3737
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9585 - loss: 0.1495 - val_accuracy: 

In [21]:
model = keras.models.load_model('binary_2gram.keras')
print(f"Test Accuracy: {model.evaluate(binary_2gram_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.8898 - loss: 0.2870
Test Accuracy: 0.894


### Word Counts vs. Word Presence

When processing text data, we can improve model performance by not just checking **whether** a word is present, but also by **counting how often** it appears. This is easy to achieve in Keras by setting the `output_mode` of the `TextVectorization` layer to `'count'`.

However, there’s a problem. Common words like `is`, `a`, and `the` tend to appear very frequently across all texts, regardless of their meaning. As a result, they may dominate the word counts and reduce the importance of more meaningful words.

---

### Why Standard Normalization Doesn't Work

A natural idea might be to normalize word counts using standard techniques—subtracting the mean and dividing by the standard deviation. But this doesn't work well with sparse text data:

- Most vectorized text consists mostly of zeros (sparse).
- Subtracting the mean turns many zeros into non-zero values.
- This causes:
  - Increased computation
  - Higher memory usage
  - Potential risk of overfitting
  - Loss of sparsity (which is bad for text data)

---

### TF-IDF to the Rescue

To handle this properly, we use **TF-IDF normalization** (Term Frequency–Inverse Document Frequency), which adjusts word counts based on how informative they are.

- `TF` (Term Frequency): How often a word appears in the current document.
- `IDF` (Inverse Document Frequency): How rare the word is across the entire dataset.

This approach:
- Downweights common, uninformative words
- Highlights rare, meaningful words
- Preserves sparsity (no subtraction, only scaling)

To use TF-IDF in Keras, simply set:

```python
TextVectorization(output_mode='tf_idf')


In [22]:
text_vectorization = TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode='count'
)

In [23]:
text_vectorization = TextVectorization(
    ngrams=2,
    max_tokens=20000,
    output_mode='tf_idf'
)

text_vectorization.adapt(text_only_train_ds)

tfidf_2gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
tfidf_2gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
tfidf_2gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)

In [24]:
model = get_model()
model.summary()
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='tfidf_2gram.keras',
        save_best_only=True
    )
]

model.fit(
    tfidf_2gram_train_ds.cache(),
    validation_data=tfidf_2gram_val_ds.cache(),
    epochs=10,
    callbacks=callbacks
)

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 20000)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │       320,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 320,033 (1.22 MB)

 Trainable params: 320,033 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.7176 - loss: 0.7349 - val_accuracy: 0.8878 - val_loss: 0.3029
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8618 - loss: 0.3255 - val_accuracy: 0.8542 - val_loss: 0.3420
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8804 - loss: 0.2809 - val_accuracy: 0.8834 - val_loss: 0.3307
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8877 - loss: 0.2583 - val_accuracy: 0.8784 - val_loss: 0.3180
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.8899 - loss: 0.2480 - val_accuracy: 0.8780 - val_loss: 0.3212
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8964 - loss: 0.2367 - val_accuracy: 0.8884 - val_loss: 0.3243
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8987 - loss: 0.2354 - val_accuracy: 0.8846 - val_loss: 0.3360
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.9037 - loss: 0.2278 - val_accuracy: 0.

In [25]:
model = keras.models.load_model('tfidf_2gram.keras')
print(f"Test Accuracy: {model.evaluate(tfidf_2gram_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.8819 - loss: 0.3105
Test Accuracy: 0.887


In [26]:
print(model.evaluate(tfidf_2gram_test_ds))

782/782 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.8814 - loss: 0.3113
[0.30242103338241577, 0.8871999979019165]


Here above we processed the whole dataset first and then used the model. But How we could handle it, if we have to deploy the model. The model should have it's own text vectorizer. Thankfully, keras is too good for us. We can make another model and just add a vectorization layer before the actual model using functional api.

In [27]:
inputs = keras.Input(shape=(1, ), dtype='string')
processed_inputs = text_vectorization(inputs)
outputs = model(processed_inputs)
inference_model = keras.Model(inputs=inputs, outputs=outputs)

import tensorflow as tf
raw_text_data = tf.convert_to_tensor([
    ["That was an excellent shit, I hate it."],
    ["Fucking awesome movie, but I almost slept."],
    ["Badly good performance, like impressively boring."],
    ["This was painfully fun."],
    ["I absolutely loved the disaster they created."],
    ["It was good, but terrible at the same time."],
    ["The food was disgustingly delicious."],
    ["I hate how much I enjoyed it."],
    ["Terribly executed, yet kind of brilliant."],
    ["Worst best experience ever."],
    ["Amazing! Just amazing!"],
    ["What a complete waste of time."],
    ["I don’t know whether I liked it or not."],
    ["It was just okay, nothing more."],
])

predictions = inference_model(raw_text_data)

for data, prediction in zip(raw_text_data, predictions):
    print(f"{data[0]} -> {float(prediction) * 100:.3f} percent positive")

b'That was an excellent shit, I hate it.' -> 68.110 percent positive
b'Fucking awesome movie, but I almost slept.' -> 51.723 percent positive
b'Badly good performance, like impressively boring.' -> 36.054 percent positive
b'This was painfully fun.' -> 52.377 percent positive
b'I absolutely loved the disaster they created.' -> 65.349 percent positive
b'It was good, but terrible at the same time.' -> 38.440 percent positive
b'The food was disgustingly delicious.' -> 51.954 percent positive
b'I hate how much I enjoyed it.' -> 76.059 percent positive
b'Terribly executed, yet kind of brilliant.' -> 50.839 percent positive
b'Worst best experience ever.' -> 49.874 percent positive
b'Amazing! Just amazing!' -> 65.122 percent positive
b'What a complete waste of time.' -> 8.184 percent positive
b'I don\xe2\x80\x99t know whether I liked it or not.' -> 59.826 percent positive
b'It was just okay, nothing more.' -> 34.252 percent positive


##  11.3.3 Processing words as a sequence: The sequence model approach

In [28]:
from tensorflow.keras import layers
from keras.layers import TextVectorization

max_length = 600
max_tokens = 20000
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode='int',
    output_sequence_length=max_length
)
text_vectorization.adapt(text_only_train_ds)

int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4
)

In [29]:
import tensorflow as tf
import math
embedding_dim = int(math.sqrt(max_tokens))

inputs = keras.Input(shape=(None,), dtype='int64')
embedded = layers.Embedding(input_dim=max_tokens, output_dim=embedding_dim)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [30]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='one_hot_bidir_lstm.keras',
        save_best_only=True
    )
]

model.fit(
    int_train_ds,
    validation_data=int_val_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 41ms/step - accuracy: 0.6183 - loss: 0.6297 - val_accuracy: 0.8036 - val_loss: 0.4719
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 40ms/step - accuracy: 0.8335 - loss: 0.4063 - val_accuracy: 0.8266 - val_loss: 0.4080
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 42s 41ms/step - accuracy: 0.8807 - loss: 0.3208 - val_accuracy: 0.8716 - val_loss: 0.3305
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 27s 43ms/step - accuracy: 0.8990 - loss: 0.2792 - val_accuracy: 0.8826 - val_loss: 0.3159
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 39s 40ms/step - accuracy: 0.9145 - loss: 0.2421 - val_accuracy: 0.7914 - val_loss: 0.8221
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 49ms/step - accuracy: 0.9259 - loss: 0.2252 - val_accuracy: 0.8856 - val_loss: 0.3180
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 36s 41ms/step - accuracy: 0.9361 - loss: 0.1863 - val_accuracy: 0.8692 - val_loss: 0.3559
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 49ms/step - accuracy: 0.9451 - loss: 0.1588 - 

In [31]:
model = keras.models.load_model('one_hot_bidir_lstm.keras')
print(f"Test Accuracy: {model.evaluate(int_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - accuracy: 0.8656 - loss: 0.3419
Test Accuracy: 0.869


### Embedding Layers

In [32]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [33]:
inputs = keras.Input(shape=(None, ), dtype='int64')
embedded = layers.Embedding(input_dim=max_tokens, output_dim=256)(inputs)
x = layers.Bidirectional(layers.LSTM(32,))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='embeddings_bidir_gru.keras',
        save_best_only=True
    )
]

model.fit(
    int_train_ds,
    validation_data=int_val_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 60ms/step - accuracy: 0.6234 - loss: 0.6262 - val_accuracy: 0.8156 - val_loss: 0.4372
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 39s 56ms/step - accuracy: 0.8319 - loss: 0.4112 - val_accuracy: 0.8354 - val_loss: 0.4083
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 56ms/step - accuracy: 0.8728 - loss: 0.3315 - val_accuracy: 0.8524 - val_loss: 0.3798
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 38s 51ms/step - accuracy: 0.9007 - loss: 0.2745 - val_accuracy: 0.8590 - val_loss: 0.4185
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 45s 58ms/step - accuracy: 0.9121 - loss: 0.2427 - val_accuracy: 0.8632 - val_loss: 0.3677
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 37s 52ms/step - accuracy: 0.9295 - loss: 0.2035 - val_accuracy: 0.8736 - val_loss: 0.3475
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 53ms/step - accuracy: 0.9446 - loss: 0.1676 - val_accuracy: 0.8646 - val_loss: 0.4353
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 34s 42ms/step - accuracy: 0.9554 - loss: 0.1405 - 

In [34]:
model = keras.models.load_model('embeddings_bidir_gru.keras')
print(f"test acc: {model.evaluate(int_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.8551 - loss: 0.3776
test acc: 0.860


In [35]:
from keras.layers import Embedding

embedding_layer = Embedding(input_dim=10, output_dim=256, mask_zero=True)
some_input = [
[4, 3, 2, 1, 0, 0, 0],
[5, 4, 3, 2, 1, 0, 0],
[2, 1, 0, 0, 0, 0, 0]]
mask = embedding_layer.compute_mask(some_input)

### Using Embedding Layer with masking enabled

In [36]:
inputs = keras.Input(shape=(None,), dtype='int64')
embedded = Embedding(input_dim=max_tokens, output_dim=256, mask_zero=True)(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='embeddings_bidir_gru_with_masking.keras',
        save_best_only=True
    )
]

model.fit(
    int_train_ds,
    validation_data=int_val_ds,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 43ms/step - accuracy: 0.6790 - loss: 0.5715 - val_accuracy: 0.7326 - val_loss: 0.6719
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 42ms/step - accuracy: 0.8650 - loss: 0.3277 - val_accuracy: 0.8828 - val_loss: 0.2991
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 26s 42ms/step - accuracy: 0.8978 - loss: 0.2633 - val_accuracy: 0.8638 - val_loss: 0.3844
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 42ms/step - accuracy: 0.9229 - loss: 0.2062 - val_accuracy: 0.8752 - val_loss: 0.3330
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 42ms/step - accuracy: 0.9373 - loss: 0.1664 - val_accuracy: 0.8796 - val_loss: 0.3814
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 42ms/step - accuracy: 0.9560 - loss: 0.1251 - val_accuracy: 0.8656 - val_loss: 0.4771
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 41ms/step - accuracy: 0.9676 - loss: 0.0920 - val_accuracy: 0.8854 - val_loss: 0.4355
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 42s 43ms/step - accuracy: 0.9791 - loss: 0.0625 - 

In [38]:
model = keras.models.load_model('embeddings_bidir_gru_with_masking.keras')
print(f"test acc: {model.evaluate(int_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 23ms/step - accuracy: 0.8702 - loss: 0.3190
test acc: 0.875


In [41]:
!wget http://nlp.stanford.edu/data/glove.6B.zip

--2025-07-23 12:28:22--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-07-23 12:28:22--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-07-23 12:28:22--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip.1’

gl

In [42]:
# !rm glove.6B.zip
# !mv glove.6B.zip.1 glove.6B.zip

In [43]:
!unzip -q glove.6B.zip

In [44]:
import numpy as np
path_to_glove_file = "glove.6B.100d.txt"

embeddings_index = {}
with open(path_to_glove_file) as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, 'f', sep=" ")
        embeddings_index[word] = coefs

print(f"Found {len(embeddings_index)} word vectors")

Found 400000 word vectors


In [45]:
embedding_dim = 100

vocabulary = text_vectorization.get_vocabulary()
word_index = dict(zip(vocabulary, range(len(vocabulary))))

embedding_matrix = np.zeros((max_tokens, embedding_dim))
for word, i in word_index.items():
    if i < max_tokens:
        embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

In [46]:
embedding_layer = layers.Embedding(
    input_dim=max_tokens,
    output_dim=embedding_dim,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False,
    mask_zero=True
)

In [48]:
inputs = keras.Input(shape=(None, ), dtype='int64')
embedded = embedding_layer(inputs)
x = layers.Bidirectional(layers.LSTM(32))(embedded)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer='rmsprop',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='glove_embeddings_sequence_model.keras',
        save_best_only=True
    )
]

model.summary()

model.fit(
    int_train_ds,
    validation_data=int_val_ds,
    epochs=10,
    callbacks=callbacks
)

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, None, 100) │  2,000,000 │ input_layer_8[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, None)      │          0 │ input_layer_8[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_4     │ (None, 64)        │     34,048 │ embedding_4[1][0… │
│ (Bidirectional)     │                   │            │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 64)        │          0 │ bidirectional_4[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 1)         │         65 │ dropout_7[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,034,113 (7.76 MB)

 Trainable params: 34,113 (133.25 KB)

 Non-trainable params: 2,000,000 (7.63 MB)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 47s 67ms/step - accuracy: 0.6315 - loss: 0.6325 - val_accuracy: 0.7794 - val_loss: 0.4720
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 72s 51ms/step - accuracy: 0.7797 - loss: 0.4737 - val_accuracy: 0.7936 - val_loss: 0.4331
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 38s 45ms/step - accuracy: 0.8224 - loss: 0.4083 - val_accuracy: 0.8334 - val_loss: 0.3798
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 39s 43ms/step - accuracy: 0.8421 - loss: 0.3720 - val_accuracy: 0.8178 - val_loss: 0.4010
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.8515 - loss: 0.3475 - val_accuracy: 0.8574 - val_loss: 0.3313
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 38s 40ms/step - accuracy: 0.8635 - loss: 0.3274 - val_accuracy: 0.8420 - val_loss: 0.3509
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 28s 44ms/step - accuracy: 0.8688 - loss: 0.3139 - val_accuracy: 0.8656 - val_loss: 0.3158
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 38s 40ms/step - accuracy: 0.8785 - loss: 0.2946 - 

In [49]:
model = keras.models.load_model('glove_embeddings_sequence_model.keras')
print(f"test acc: {model.evaluate(int_test_ds)[1]:.3f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.8652 - loss: 0.3251
test acc: 0.869
